# modeling_v13 — M4 보조: SegLGBM(레짐 세그) · XGBoost 정직축 점수 (도전자 기록)

**목적** — 듀얼(LGBM·ET) 위에 **SegLGBM · XGBoost** 두 도전자의 **GKF(C20)** 점수를 같은 셋·같은 v2.2 stable 폴드맵으로 추가 기록. **B1_LGBM(공식 기준선)은 불변** — 두 모델은 도전자 기록이며, **정직축(GKF)에서 LGBM 을 이겨야** M5 §6.2 챔피언 후보 승격을 논의한다(다중시드 확인 필수, **KFold 우위는 불인정 R1**). 듀얼 노트북과 **병렬 실행** 가능.

> **전제 파일** (듀얼과 동일, 자동 탐색): `v13_fdc_pool_wf_oof.csv.gz` · `core10_meta_wf.csv` · `feature_diet_selected.json` · `select_result_Conservative_GA.json` · `select_result_Balanced_GA.json`
>
> **런타임(로컬 CPU)** : 2셋 × 2모델 × (KF5+GKF5) = **40 fit** ≈ **~30분**. `Restart & Run All`.

### 모델 정의 (누수 안전 — 강건 점검 완료)
- **SegLGBM (레짐 세그)** : `is_high_regime`(물리 PM-loud 레짐)로 **fold 내 train 을 세그 분할** → 세그별 LGBM(`M8_PARAMS`·705) 학습 → val 을 **자기 레짐 모델로 라우팅**. 세그 train 이 `SEG_MIN_ROWS` 미만이면 글로벌 폴백. **세그 기준 = 피처값(레짐)이지 lot/WF 가 아니므로 GKF 누수 없음**(val lot 은 여전히 train 미포함). 분포 seg0 3,692 / seg1 8,247, fold별 각 ~290/680 lot 로 안정.
- **XGBoost** : `tree_method='hist'`, **NaN 자체 처리(대치 없음 → LGBM 과 동일 조건, ET 의 median 왜곡 없음)**, 미튜닝 M8 계열 근사 파라미터(도전자). seed 42 결정적(2회 동일 확인).
- 정직축 = **동결 stable 폴드맵(v2.2)** — 로컬·미러 어디서든 동일 파티션. **KFold 는 기록만(R1)**.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, re, json, time
import numpy as np, pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# ── 동결 상수 ──
ID_COL, TARGET_COL = "C64", "C65"
CORE10 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
          "dslp_x_hour", "hour", "hour_x_c33",
          "C60_mean_step4", "C59_mean_step4", "is_special_recipe"]
M8_PARAMS = dict(objective="regression", metric="rmse",
    learning_rate=0.029017547696366934, num_leaves=175, min_child_samples=5,
    feature_fraction=0.6324704159196377, bagging_fraction=0.864012693783303, bagging_freq=7,
    lambda_l1=5.04154328625296, lambda_l2=0.024814259264649002,
    min_split_gain=0.2573073648505903, verbose=-1, seed=42)
BEST_ROUNDS = 705

# ── SegLGBM ──
SEG_COL = "is_high_regime"      # 물리 PM-loud 레짐 (세그 기준; lot/WF 아님)
SEG_MIN_ROWS = 300              # 세그 train 이 이 미만이면 글로벌 폴백

# ── XGBoost (미튜닝 M8 계열 근사; NaN 자체 처리) ──
XGB_PARAMS = dict(n_estimators=BEST_ROUNDS, learning_rate=0.029, max_depth=8,
                  subsample=0.86, colsample_bytree=0.63, min_child_weight=5,
                  reg_lambda=0.02, reg_alpha=5.0, gamma=0.0,
                  tree_method="hist", random_state=42, n_jobs=-1, verbosity=0)

PROTECTED = ["C17", "C11", "C31", "C15", "C16"]
B0_REF = 71.212
SIGMA_C65 = 261.7
REF_LGBM_GKF = {"Conservative": 71.366, "Balanced": 71.272}   # 듀얼(로컬 동결) 기준선 — 도전자 대조용
REF_LGBM_KF  = {"Conservative": 50.677, "Balanced": 51.412}

def _find(name):
    for d in [".", "data", "colab_GA", "..",
              os.path.join("..", "modeling_v13"),
              os.path.join("..", "modeling_v13", "data"),
              os.path.join("..", "modeling_v13", "colab_GA")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음 — 노트북과 같은 폴더(또는 data/·colab_GA/)에 두세요.")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz")
META = _find("core10_meta_wf.csv")
DIET = _find("feature_diet_selected.json")
SELJSON = {p: _find(f"select_result_{p}_GA.json") for p in ["Conservative", "Balanced"]}
print("파일 확인:", *[os.path.basename(x) for x in [POOL, META, DIET, *SELJSON.values()]])

파일 확인: v13_fdc_pool_wf_oof.csv.gz core10_meta_wf.csv feature_diet_selected.json select_result_Conservative_GA.json select_result_Balanced_GA.json


In [2]:
# ── 헬퍼 ──
def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def sensor_of(c):
    m = re.match(r"(C\d+)_", c)
    return m.group(1) if m else c

def floor_ok(feats):
    have = {s: sum(1 for c in feats if sensor_of(c) == s) for s in PROTECTED}
    return all(v >= 1 for v in have.values()), have

# 환경독립 GroupKFold (v2.2 동결 stable 폴드맵 — 듀얼과 동일)
def stable_group_kfold(groups, n_splits=5):
    groups = np.asarray(groups)
    _, inv, cnt = np.unique(groups, return_inverse=True, return_counts=True)
    order = np.argsort(cnt, kind="stable")[::-1]
    fold_sizes = np.zeros(n_splits, dtype=np.int64); g2f = np.zeros(len(cnt), dtype=np.int64)
    for gi in order:
        f = int(np.argmin(fold_sizes)); fold_sizes[f] += cnt[gi]; g2f[gi] = f
    return g2f[inv]

def _gkf_splits(groups, n_splits=5):
    fv = stable_group_kfold(groups, n_splits)
    for k in range(n_splits):
        yield np.where(fv != k)[0], np.where(fv == k)[0]

def _kf_splits(df):
    for k in range(5):
        yield np.where((df["fold_kf5"] != k).to_numpy())[0], np.where((df["fold_kf5"] == k).to_numpy())[0]

def miss_frac(df, feats):
    std_cols = [c for c in feats if "_std_" in c]; s5 = [c for c in feats if c.endswith("step5")]
    return dict(all=round(float(df[feats].isna().mean().mean()), 4),
                std=round(float(df[std_cols].isna().mean().mean()), 4) if std_cols else 0.0,
                step5=round(float(df[s5].isna().mean().mean()), 4) if s5 else 0.0)

# ── SegLGBM OOF (fold-내 세그별 학습 → 자기 레짐 라우팅; 누수 안전) ──
def _seg_oof(splits, df, feats, y):
    seg = df[SEG_COL].to_numpy(); oof = np.zeros(len(df))
    for tr, va in splits:
        Xtr, ytr, s_tr = df.iloc[tr][feats], y[tr], seg[tr]
        Xva, s_va = df.iloc[va][feats], seg[va]
        uniq = list(np.unique(s_tr))
        models = {s: (lgb.train(M8_PARAMS, lgb.Dataset(Xtr[s_tr == s], ytr[s_tr == s]), num_boost_round=BEST_ROUNDS)
                      if (s_tr == s).sum() >= SEG_MIN_ROWS else None) for s in uniq}
        need_glob = any(m is None for m in models.values()) or bool(set(np.unique(s_va)) - set(uniq))
        glob = lgb.train(M8_PARAMS, lgb.Dataset(Xtr, ytr), num_boost_round=BEST_ROUNDS) if need_glob else None
        pred = np.empty(len(va))
        for s in np.unique(s_va):
            mdl = models.get(s); mdl = mdl if mdl is not None else glob
            pred[s_va == s] = mdl.predict(Xva[s_va == s])
        oof[va] = pred
    return _rmse(y, oof)

def oof_group_seg(df, feats, y, groups):
    return _seg_oof(_gkf_splits(groups), df, feats, y)
def oof_kfold_seg(df, feats, y):
    return _seg_oof(_kf_splits(df), df, feats, y)

# ── XGBoost OOF (NaN 자체 처리, 대치 없음) ──
def _xgb_oof(splits, df, feats, y):
    oof = np.zeros(len(df))
    for tr, va in splits:
        m = xgb.XGBRegressor(**XGB_PARAMS).fit(df.iloc[tr][feats], y[tr])
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)

def oof_group_xgb(df, feats, y, groups):
    return _xgb_oof(_gkf_splits(groups), df, feats, y)
def oof_kfold_xgb(df, feats, y):
    return _xgb_oof(_kf_splits(df), df, feats, y)

In [3]:
# ── 로드 & 두 셋 복원 (듀얼과 동일 구조) ──
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")
y = df[TARGET_COL].to_numpy(float); groups = df["C20"].to_numpy()
diet = json.loads(open(DIET, encoding="utf-8").read())
assert SEG_COL in df.columns, "세그 기준 컬럼 없음"

SETS = {}
for p in ["Conservative", "Balanced"]:
    champions = list(diet["champions"][p].values())
    fixed = [f for f in dict.fromkeys(CORE10 + champions) if f in df.columns]
    selp = json.loads(open(SELJSON[p], encoding="utf-8").read())["selected_features"]
    feats = list(dict.fromkeys(fixed + selp))
    ok, have = floor_ok(feats)
    assert all(f in df.columns for f in feats) and ok, f"{p}: 피처/floor 문제 {have}"   # R10
    SETS[p] = dict(feats=feats, n=len(feats), floor=have, miss=miss_frac(df, feats))
    print(f"{p:12s} n={len(feats):3d}  floor={have}")
vc = pd.Series(df[SEG_COL]).value_counts().to_dict()
print(f"df {df.shape} | unique C20 {len(np.unique(groups))} | 레짐 세그 {SEG_COL} 분포 {vc}")

Conservative n= 99  floor={'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
Balanced     n=108  floor={'C17': 2, 'C11': 9, 'C31': 2, 'C15': 3, 'C16': 3}
df (11939, 667) | unique C20 1217 | 레짐 세그 is_high_regime 분포 {1: 8247, 0: 3692}


In [4]:
# ── 2셋 × 2모델 × (KFold5 + GKF5) = 40 fit ──  (로컬 ~30분)
rows = []
for p in ["Conservative", "Balanced"]:
    feats = SETS[p]["feats"]
    for model, kf_fn, gk_fn in [("SegLGBM", oof_kfold_seg, oof_group_seg),
                                 ("XGBoost", oof_kfold_xgb, oof_group_xgb)]:
        t = time.time()
        kf = kf_fn(df, feats, y); gk = gk_fn(df, feats, y, groups)
        dt = time.time() - t
        rows.append(dict(candidate=f"{p}-GA", model=model, n_total=len(feats),
                         KFold_OOF=round(kf, 3), GroupKFold_C20=round(gk, 3),
                         floor_ok=True, miss_all=SETS[p]["miss"]["all"], sec=round(dt, 0)))
        print(f"{p:12s} {model:8s}  KFold={kf:.3f}  GroupKFold(C20)={gk:.3f}  ({dt:.0f}s)")

res = pd.DataFrame(rows, columns=["candidate", "model", "n_total", "KFold_OOF",
                                  "GroupKFold_C20", "floor_ok", "miss_all", "sec"])
res

Conservative SegLGBM   KFold=50.707  GroupKFold(C20)=71.289  (2457s)
Conservative XGBoost   KFold=52.259  GroupKFold(C20)=70.773  (137s)
Balanced     SegLGBM   KFold=51.193  GroupKFold(C20)=71.527  (1509s)
Balanced     XGBoost   KFold=52.568  GroupKFold(C20)=70.651  (138s)


,candidate,model,n_total,KFold_OOF,GroupKFold_C20,floor_ok,miss_all,sec
0,Conservative-GA,SegLGBM,99,50.707,71.289,True,0.1306,2457.0
1,Conservative-GA,XGBoost,99,52.259,70.773,True,0.1306,137.0
2,Balanced-GA,SegLGBM,108,51.193,71.527,True,0.0248,1509.0
3,Balanced-GA,XGBoost,108,52.568,70.651,True,0.0248,138.0


In [5]:
# ── 정직축 요약 (도전자 vs LGBM 기준선) ──
piv = res.pivot(index="candidate", columns="model", values="GroupKFold_C20")
print("정직축 GKF(C20) — 도전자 기록 (기준선 = LGBM 듀얼 로컬 동결)\n")
summary = {}
for p in ["Conservative", "Balanced"]:
    lg = REF_LGBM_GKF[p]
    seg = float(piv.loc[f"{p}-GA", "SegLGBM"]); xg = float(piv.loc[f"{p}-GA", "XGBoost"])
    print(f"  {p:12s}  LGBM(기준) {lg:.3f}  |  SegLGBM {seg:.3f} ({seg-lg:+.3f})  |  XGBoost {xg:.3f} ({xg-lg:+.3f})")
    summary[p] = dict(LGBM_ref=lg, SegLGBM=round(seg, 3), XGBoost=round(xg, 3),
                      SegLGBM_delta=round(seg-lg, 3), XGBoost_delta=round(xg-lg, 3))
best_gkf = min([(float(r["GroupKFold_C20"]), r["candidate"], r["model"]) for _, r in res.iterrows()])
print(f"\n이 노트북 최저 GKF: {best_gkf[2]} on {best_gkf[1]} = {best_gkf[0]:.3f}")
print("판독: 정직축에서 LGBM(기준선)을 유의하게(≥0.3) 이긴 모델만 M5 §6.2 챔피언 후보 승격을 논의(다중시드 확인). KFold 우위는 불인정(R1).")
print("     ※ 공식 셋 확정·B1 동결은 듀얼 노트북 결과로. 본 노트북은 도전자 기록.")

정직축 GKF(C20) — 도전자 기록 (기준선 = LGBM 듀얼 로컬 동결)

  Conservative  LGBM(기준) 71.366  |  SegLGBM 71.289 (-0.077)  |  XGBoost 70.773 (-0.593)
  Balanced      LGBM(기준) 71.272  |  SegLGBM 71.527 (+0.255)  |  XGBoost 70.651 (-0.621)

이 노트북 최저 GKF: XGBoost on Balanced-GA = 70.651
판독: 정직축에서 LGBM(기준선)을 유의하게(≥0.3) 이긴 모델만 M5 §6.2 챔피언 후보 승격을 논의(다중시드 확인). KFold 우위는 불인정(R1).
     ※ 공식 셋 확정·B1 동결은 듀얼 노트북 결과로. 본 노트북은 도전자 기록.


In [6]:
# ── 저장 ──
out = dict(results=rows, summary_vs_lgbm=summary,
           ref=dict(LGBM_GKF=REF_LGBM_GKF, LGBM_KF=REF_LGBM_KF, B0=B0_REF),
           models=dict(SegLGBM=f"is_high_regime 세그 · M8_PARAMS·{BEST_ROUNDS} · 폴백<{SEG_MIN_ROWS}행",
                       XGBoost={k: XGB_PARAMS[k] for k in ["n_estimators","learning_rate","max_depth","subsample","colsample_bytree"]}),
           cv="stable_group_kfold (v2.2)",
           note="도전자 기록. 공식 기준선=B1_LGBM(듀얼). KFold 우위 불인정(R1).")
json.dump(out, open("compare_seg_xgb_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
res.to_csv("compare_seg_xgb_results.csv", index=False)
print("saved: compare_seg_xgb_results.json / compare_seg_xgb_results.csv")

saved: compare_seg_xgb_results.json / compare_seg_xgb_results.csv


---
### 판정 & 합치기
- 이 노트북은 **도전자 기록**(SegLGBM·XGBoost). **공식 셋 확정·B1_LGBM 동결은 듀얼 노트북** 결과로 강건이 내린다.
- 정직축(GKF)에서 **LGBM 기준선을 유의하게(≥0.3pt) 이긴** 모델이 있으면 → M5 §6.2 **챔피언 후보 승격**을 다중시드 lot-CV 확인과 함께 논의(그 경우 §6.2 개정 = 사용자 승인).
- **KFold 우위는 채택 근거 불인정**(R1 — lot-mate 암기 배제). ET·SegLGBM·XGBoost 모두 동일 규율.
- 합산 보고: 듀얼 4행 + 본 노트북 4행 = **8행(2셋 × 4모델)** 정직축 표를 강건이 REPORT 에 정리.